<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "480"
iso_country = "Mauritius"
year = 2024
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 14.950*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Mauritius': 

Number of trade records present in dataset: 31486
Number of partner countries present in dataset: 170
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$5724.5m
Total weight within the dataset: 6049.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,480,Mauritius,724,Spain,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",20608.610,NaN,20608.610,kg,1302.0,1103.0,3433.394426,11013.0,6843.158823,23495.263359,['Container'],['Container'],69.044268,68.866517,68.885619,68.885619,68.968647,68.968647,72.610252,133.871708,78.860545,74.041631,74.320838,76.327554,70.938169,130.621866,84.005026,77.024345,74.993866,78.917363,69.470058,145.421777,103.556543,91.881500,84.536002,82.524875,68.029881,152.037208,123.540612,113.878744,91.272980,88.922143,66.617998,158.333257,153.502292,152.930177,97.665609,93.670401,65.178229,154.595147,148.573503,160.875159,125.037799,95.427730,63.761649,134.804176,144.113890,158.519078,117.392965,96.565734,62.380537,120.807902,123.170429,146.920259,118.850602,97.284940,3693
1,1,480,Mauritius,784,United Arab Emirates,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",177.043,NaN,177.043,kg,34.0,34.0,29.495364,6029.0,3746.245759,110.496882,['Container'],['Container'],0.324711,0.323875,0.323965,0.323965,0.324356,0.324356,0.341482,0.629591,0.370877,0.348214,0.349527,0.358964,0.333618,0.614307,0.395071,0.362241,0.352692,0.371144,0.326714,0.683910,0.487021,0.432113,0.397568,0.388110,0.319941,0.715022,0.581004,0.535565,0.429252,0.418196,0.313301,0.744632,0.721913,0.719222,0.459316,0.440527,0.306529,0.727052,0.698733,0.756587,0.588046,0.448791,0.299867,0.633976,0.677759,0.745506,0.552092,0.454143,0.293372,0.568153,0.579263,0.690958,0.558948,0.457526,3693


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1286.4m (22.5%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
03 | Fish and crustaceans  203225360.4   86325599.1      3.6      1.4
10 | Cereals               146938099.2  418453559.6      2.6      6.9
04 | Dairy produce; birds  129002585.0   46864055.3      2.3      0.8
19 | Preparations of cere   93566102.8   33418813.5      1.6      0.6
21 | Miscellaneous edible   93386079.0   24089684.4      1.6      0.4
22 | Beverages, spirits a   77217507.6  130582556.7      1.3      2.2
15 | Animal, vegetable or   67041945.6   52590207.9      1.2      0.9
02 | Meat and edible meat   64467368.7   13709814.0      1.1      0.2
16 | Meat, fish, crustace   60141655.6   13073100.4      1.1      0.2
23 | Food industries, res   57036209.2   89909374.6      1.0      1.5
17 | Sugars and sugar con   56065527.6  101321588.8      1

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1122.5m (19.6%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  522172377.3  339940661.8      9.1      5.6
Agriculture       460814621.2  490664467.6      8.0      8.1
Fishing           139515305.9   64346609.2      2.4      1.1



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1028.9m (18.0%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
OFD - Food products n.e.c.                    496808526.1  221119995.7   
MIL - Dairy products                          126304260.6   40199211.0   
VOL - Vegetable oils and fats                  88071023.5  118344788.9   
V_F - Vegetables, fruit, nuts                  71258984.6  102677894.2   
PCR - Processed rice                           67990307.5  118099164.2   
CMT - Bovine meat prods                        56637981.1   10343548.6   
OMT - Meat products n.e.c.                     44899516.2   11789195.1   
PFB - Plant-based fibers                       34998173.0   18019245.3   
GRO - Cereal grains n.e.c.                     29461021.5  121756020.0   
OCR - Crops n.e.c.                              6787581.0    3514665.8   
OAP - Animal products n.e.c.                 

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 22.5% 
GTAP 18.0% 
EORA 19.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46397/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,480,Mauritius,724,Spain,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",20608.610,NaN,20608.610,kg,1302.0,1103.0,3433.394426,11013.0,6843.158823,23495.263359,['Container'],['Container'],69.044268,68.866517,68.885619,68.885619,68.968647,68.968647,72.610252,133.871708,78.860545,74.041631,74.320838,76.327554,70.938169,130.621866,84.005026,77.024345,74.993866,78.917363,69.470058,145.421777,103.556543,91.881500,84.536002,82.524875,68.029881,152.037208,123.540612,113.878744,91.272980,88.922143,66.617998,158.333257,153.502292,152.930177,97.665609,93.670401,65.178229,154.595147,148.573503,160.875159,125.037799,95.427730,63.761649,134.804176,144.113890,158.519078,117.392965,96.565734,62.380537,120.807902,123.170429,146.920259,118.850602,97.284940,3693,NaN
1,1,480,Mauritius,784,United Arab Emirates,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",177.043,NaN,177.043,kg,34.0,34.0,29.495364,6029.0,3746.245759,110.496882,['Container'],['Container'],0.324711,0.323875,0.323965,0.323965,0.324356,0.324356,0.341482,0.629591,0.370877,0.348214,0.349527,0.358964,0.333618,0.614307,0.395071,0.362241,0.352692,0.371144,0.326714,0.683910,0.487021,0.432113,0.397568,0.388110,0.319941,0.715022,0.581004,0.535565,0.429252,0.418196,0.313301,0.744632,0.721913,0.719222,0.459316,0.440527,0.306529,0.727052,0.698733,0.756587,0.588046,0.448791,0.299867,0.633976,0.677759,0.745506,0.552092,0.454143,0.293372,0.568153,0.579263,0.690958,0.558948,0.457526,3693,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $732.6m of $5724.5m (12.8%)
Volume: 821.9 of 6049.3 million kilograms (13.6%)
Value-to-Weight: 0.89 from 0.95 US$ per kg (+-5.8%)
Average distance to market (volume-weighted): 5606.1 from 4139.0 nm (+35.4%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    945  268820501.7  194779688.1      4.7      3.2
1                Dairy    191  127228981.6   46334512.6      2.2      0.8
2  Unprocessed Cereals     30   78663647.5  299860837.0      1.4      5.0
3        Fats and Oils    150   62680560.4   47761980.4      1.1      0.8
4           Vegetables    362   56648167.9   92663873.6      1.0      1.5
5      Sugar and Honey    132   56282238.0  101383719.7      1.0      1.7
6        Meat and Prep     59   44454874.4    9079946.4      0.8      0.2
7               Fruits    241   37829431.0   30052472.3      0.7      0.5



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,92509703.6,170164930.7,100390814.5,94308871.2,94660362.5,97208994.4,84925600.1,2.012292e+08,1.950031e+08,1.943883e+08,124147917.8,119226326.1,79544159.4,153717351.4,156735690.1,1.867974e+08,151121305.1,123878010.3
1,Fats and Oils,90252121.6,155964610.6,96881573.0,91030833.4,91335701.3,94239076.2,83823723.9,1.797519e+08,1.754126e+08,1.785772e+08,118086870.4,112770098.5,79295851.5,143395209.8,145828890.5,1.711211e+08,140103786.0,116651195.5
2,Fruits,27154707.9,49474749.7,29405659.4,27648190.2,27750720.6,28505330.2,24973798.3,5.851818e+07,5.658525e+07,5.655799e+07,36328498.2,34838628.4,23425925.9,44958701.3,45865850.8,5.442093e+07,44065165.7,36181950.4
3,Meat and Prep,16072461.6,29632838.7,17455979.6,16389300.9,16451104.2,16895295.7,14746061.1,3.504746e+07,3.397812e+07,3.385148e+07,21618527.9,20734178.4,13808088.5,26741132.6,27264083.8,3.252117e+07,26307879.2,21534265.8
4,Processed Cereals,243361733.9,441643481.5,263293045.1,247531695.7,248467601.5,255352225.3,223976188.3,5.224326e+08,5.052527e+08,5.055448e+08,325613058.9,311496910.2,210232256.2,401830902.7,410340398.3,4.860869e+08,394467208.1,323309849.4
5,Sugar and Honey,131367037.4,208832379.1,138110911.9,130617278.4,131180938.8,135990184.5,123686299.2,2.467778e+08,2.353636e+08,2.440227e+08,171738366.5,157232744.5,118313377.5,204181451.9,212238426.3,2.361313e+08,199044939.6,161238399.5
6,Unprocessed Cereals,525536910.6,943799664.0,567343611.3,533372448.4,535438809.5,550861399.9,484543154.2,1.116077e+09,1.079148e+09,1.082289e+09,703226772.3,668733208.8,455603888.7,863550214.2,883523519.7,1.041031e+09,848237283.3,692958499.2
7,Vegetables,152234818.3,279359173.6,165152284.4,155117745.6,155703044.0,159941295.2,139802597.0,3.304166e+08,3.201025e+08,3.193164e+08,204451755.2,195929804.3,131003421.9,252704532.0,257767568.1,3.068471e+08,248462016.2,203446980.5


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.61,0.06,0.01,0.02,0.04,0.91,0.87,0.86,0.31,0.27,0.58,0.61,0.84,0.56,0.35
1,Fats and Oils,1.05,0.11,0.01,0.02,0.06,1.53,1.46,1.51,0.55,0.46,1.02,1.06,1.46,0.97,0.60
2,Fruits,0.59,0.06,0.01,0.02,0.04,0.89,0.84,0.83,0.30,0.26,0.57,0.59,0.82,0.55,0.34
3,Meat and Prep,0.31,0.03,0.01,0.01,0.02,0.46,0.43,0.43,0.15,0.13,0.29,0.30,0.42,0.28,0.17
4,Processed Cereals,0.74,0.07,0.02,0.02,0.04,1.11,1.05,1.05,0.38,0.33,0.71,0.74,1.03,0.69,0.42
5,Sugar and Honey,1.38,0.12,-0.01,-0.00,0.08,2.19,1.98,2.14,0.85,0.60,1.53,1.67,2.09,1.43,0.76
6,Unprocessed Cereals,5.32,0.53,0.10,0.13,0.32,8.03,7.56,7.60,2.78,2.34,5.19,5.44,7.44,4.99,3.02
7,Vegetables,2.24,0.23,0.05,0.06,0.14,3.36,3.18,3.17,1.14,0.99,2.15,2.24,3.10,2.07,1.28
